# Nutrition5k: resultados

No entrena nada: lee los artefactos guardados en `outputs/` (configuración, historial por época, métricas de validación y test)
y los resume. Funciona en local, en Colab (lee de Drive) y en un pod de RunPod (lee del volumen a través del enlace `outputs/`).

Elegir siempre por **validación** (`val_mean_pmae`); test se informa pero no decide.

In [ ]:
import json
import os
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nutrition5k")
except ImportError:
    if "RUNPOD_POD_ID" in os.environ:
        # The clone made by 02_experiments.ipynb: outputs/ links to the volume and data/ holds the frames.
        os.chdir("/root/thesis-multimodal-health-ai")
    elif Path.cwd().name == "notebooks":
        os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

SPLIT = "full"         # "full" o "pilot"
MODE = "complete"      # modo de frames a comparar
TARGETS = ("total_mass", "total_calories", "total_fat", "total_carb", "total_protein")
SERIES = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
INK, MUTED = "#0b0b0b", "#52514e"

In [ ]:
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None


def load_runs():
    rows = []
    for config_path in sorted(Path("outputs/mobilenet").glob("*/config.json")):
        run = config_path.parent
        config = read_json(config_path)
        history = pd.read_csv(run / "history.csv") if (run / "history.csv").is_file() else None
        val, test = read_json(run / "evaluation_val.json"), read_json(run / "evaluation.json")
        row = {
            "run": run.name,
            "backbone": config["backbone"],
            "unfreeze": "all" if config["unfreeze"] is None else str(config["unfreeze"]),
            "heads": config["heads"],
            "hidden": "x".join(map(str, config["hidden"])) or "0",
            "image_size": config["image_size"],
            "seed": config["seed"],
            "split": config["dataset_mode"],
            "mode": config["frame_mode"],
            "params_M": config["parameters"] / 1e6,
            "trainable_M": config["trainable_parameters"] / 1e6,
            "epochs_done": 0 if history is None else len(history),
            "best_epoch": None if history is None else int(history["val_mean_pmae"].idxmin()),
            "sec_per_epoch": None if history is None else history["seconds"].mean(),
            # Older runs predate these fields.
            "gpu": config.get("environment", {}).get("gpu"),
            "gpu_memory_gb": history["gpu_memory_gb"].max() if history is not None and "gpu_memory_gb" in history else None,
            "status": "done" if test else "running",
            "val_mean_pmae": val["mean_pmae"] if val else None,
            "test_mean_pmae": test["mean_pmae"] if test else None,
        }
        for target in TARGETS:
            if test:
                row[f"{target}_pmae"] = test["metrics"][target]["pmae"]
                row[f"{target}_mae"] = test["metrics"][target]["mae"]
        rows.append(row)
    return pd.DataFrame(rows)


runs = load_runs()
runs = runs[(runs["split"] == SPLIT) & (runs["mode"] == MODE)] if len(runs) else runs
baseline = read_json(Path(f"outputs/average-baseline/{SPLIT}/evaluation.json"))
BASELINE_PMAE = baseline["mean_pmae"] if baseline else None
print(f"{len(runs)} runs; baseline mean PMAE (test): {BASELINE_PMAE}")
runs.sort_values("val_mean_pmae")

In [ ]:
def select(**fixed):
    """Runs matching every given field, e.g. select(backbone="v4s", heads="single")."""
    chosen = runs
    for field, value in fixed.items():
        chosen = chosen[chosen[field].astype(str) == str(value)]
    return chosen.sort_values("val_mean_pmae")


def bar_pmae(chosen, label, title):
    """Validation mean PMAE per run (lower is better), with the test baseline as reference."""
    done = chosen.dropna(subset=["val_mean_pmae"]).sort_values("val_mean_pmae", ascending=False)
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(done) + 1.2))
    bars = ax.barh(done[label].astype(str), done["val_mean_pmae"], height=0.6, color=SERIES[0])
    ax.bar_label(bars, fmt="%.1f%%", padding=4, color=INK, fontsize=9)
    if BASELINE_PMAE:
        ax.axvline(BASELINE_PMAE, color=MUTED, linestyle="--", linewidth=1)
        ax.text(BASELINE_PMAE, len(done) - 0.4, " baseline (test)", color=MUTED, fontsize=8, va="bottom")
    ax.set_xlabel("Validation mean PMAE (%)", color=MUTED)
    ax.set_title(title, loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color="#e6e6e3", linewidth=0.8)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


def curves(chosen, label, title):
    """Validation mean PMAE per epoch; at most eight runs per chart."""
    fig, ax = plt.subplots(figsize=(7, 4))
    # Colour follows the label value (sorted), not the ranking, so it is stable across views.
    colors = dict(zip(sorted(chosen[label].astype(str).unique()), SERIES))
    for _, row in chosen.sort_values(label, key=lambda values: values.astype(str)).iterrows():
        if str(row[label]) not in colors:
            continue
        history = pd.read_csv(Path("outputs/mobilenet") / row["run"] / "history.csv")
        ax.plot(history["epoch"], history["val_mean_pmae"], color=colors[str(row[label])],
                linewidth=2, marker="o", markersize=5, label=str(row[label]))
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.set_xlabel("Epoch", color=MUTED)
    ax.set_ylabel("Validation mean PMAE (%)", color=MUTED)
    ax.set_title(title, loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e6e3", linewidth=0.8)
    ax.legend(frameon=False, title=label)
    plt.tight_layout()
    plt.show()


SUMMARY = ["run", "status", "epochs_done", "best_epoch", "val_mean_pmae", "test_mean_pmae", "params_M", "trainable_M", "sec_per_epoch"]

## Etapa 1: backbones (fine-tuning completo, cabeza `single`, 512, 224 px, semilla 0)

In [ ]:
stage1 = select(unfreeze="all", heads="single", hidden="512", image_size=224, seed=0)
display(stage1[SUMMARY + ["backbone"]])
bar_pmae(stage1, "backbone", "Stage 1: backbone comparison")
curves(stage1, "backbone", "Stage 1: validation curves")

## Etapa 2: ablaciones (rellenar con los mismos `BEST_*` que en `02_experiments.ipynb`)

In [ ]:
BEST_BACKBONE = "v4s"
BEST_UNFREEZE = "all"
BEST_HEADS = "single"
BEST_HIDDEN = "512"

steps = {
    "Step 1: fine-tuning depth": ("unfreeze", select(backbone=BEST_BACKBONE, heads="single", hidden="512", image_size=224, seed=0)),
    "Step 2: head grouping": ("heads", select(backbone=BEST_BACKBONE, unfreeze=BEST_UNFREEZE, hidden="512", image_size=224, seed=0)),
    "Step 3: trunk size": ("hidden", select(backbone=BEST_BACKBONE, unfreeze=BEST_UNFREEZE, heads=BEST_HEADS, image_size=224, seed=0)),
    "Step 4: image size": ("image_size", select(backbone=BEST_BACKBONE, unfreeze=BEST_UNFREEZE, heads=BEST_HEADS, hidden=BEST_HIDDEN, seed=0)),
}
for title, (label, chosen) in steps.items():
    print(title)
    display(chosen[SUMMARY + [label]])
    if len(chosen):
        bar_pmae(chosen, label, title)

## Error por objetivo (test) de todas las ejecuciones terminadas

In [ ]:
per_target = ["run", "val_mean_pmae"] + [f"{target}_pmae" for target in TARGETS] + [f"{target}_mae" for target in TARGETS]
runs[runs["status"] == "done"].sort_values("val_mean_pmae")[per_target]

## Evolución por objetivo de una ejecución

Comprueba si un objetivo empeora mientras la media mejora. La línea negra es la media; la vertical marca la época elegida.

In [ ]:
def target_curves(run):
    history = pd.read_csv(Path("outputs/mobilenet") / run / "history.csv")
    fig, ax = plt.subplots(figsize=(7, 4))
    for color, target in zip(SERIES, TARGETS):
        ax.plot(history["epoch"], history[f"val_{target}_pmae"], color=color, linewidth=2,
                marker="o", markersize=5, label=target.removeprefix("total_"))
    ax.plot(history["epoch"], history["val_mean_pmae"], color=INK, linewidth=2, label="mean")
    ax.axvline(history["val_mean_pmae"].idxmin(), color=MUTED, linestyle="--", linewidth=1)
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.set_xlabel("Epoch", color=MUTED)
    ax.set_ylabel("Validation PMAE (%)", color=MUTED)
    ax.set_title(run, loc="left", color=INK, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e6e3", linewidth=0.8)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


RUN = runs.sort_values("val_mean_pmae")["run"].iloc[0]  # o escribir el nombre de una ejecución
target_curves(RUN)

## Etapa 3: media ± SD entre semillas

In [ ]:
config_columns = ["backbone", "unfreeze", "heads", "hidden", "image_size"]
done = runs[runs["status"] == "done"]
seeds = done.groupby(config_columns).agg(
    seeds=("seed", "nunique"),
    val_mean=("val_mean_pmae", "mean"),
    val_sd=("val_mean_pmae", "std"),
    test_mean=("test_mean_pmae", "mean"),
    test_sd=("test_mean_pmae", "std"),
)
seeds[seeds["seeds"] > 1].sort_values("val_mean")

## Métricas de test de los modelos finales

MAE, PMAE, RMSE y R² por objetivo, con la línea base como referencia. Test solo se informa: los modelos se eligen por validación.
Por defecto muestra los tres mejores runs por validación; se puede escribir la lista a mano.

In [ ]:
FINAL_RUNS = list(runs[runs["status"] == "done"].sort_values("val_mean_pmae")["run"].head(3))


def test_metrics(run_names):
    """One row per model and target, from each run's evaluation.json and the baseline's."""
    sources = {name: Path("outputs/mobilenet") / name / "evaluation.json" for name in run_names}
    sources["average baseline"] = Path(f"outputs/average-baseline/{SPLIT}/evaluation.json")
    rows = []
    for model, path in sources.items():
        evaluation = read_json(path)
        if not evaluation:
            continue
        for target in TARGETS:
            metric = evaluation["metrics"][target]
            rows.append({
                "model": model, "target": target.removeprefix("total_"), "unit": metric["unit"],
                "MAE": metric["mae"], "PMAE (%)": metric["pmae"], "RMSE": metric["rmse"], "R²": metric["rsquared"],
            })
        rows.append({"model": model, "target": "mean", "PMAE (%)": evaluation["mean_pmae"]})
    return pd.DataFrame(rows).set_index(["model", "target"])


test_metrics(FINAL_RUNS).round(2)

### Media ± SD entre semillas (test)

Lee `outputs/mobilenet/seeds/*.json`, que escribe `train --all-seeds`.

In [ ]:
rows = []
for path in sorted(Path("outputs/mobilenet/seeds").glob("*.json")):
    if f"-{MODE}-{SPLIT}" not in path.stem:
        continue
    summary = read_json(path)
    row = {"configuration": path.stem}
    for split, prefix in (("validation", "val"), ("test", "test")):
        mean_pmae = summary[split]["mean_pmae"]
        row[f"{prefix}_mean_pmae"] = f"{mean_pmae['mean']:.2f} ± {mean_pmae['sd']:.2f}"
    for target in TARGETS:
        pmae = summary["test"]["metrics"][target]["pmae"]
        row[f"test_{target.removeprefix('total_')}_pmae"] = f"{pmae['mean']:.2f} ± {pmae['sd']:.2f}"
    rows.append(row)
pd.DataFrame(rows)

## Galería de platos de test

Para un run: los platos con menor error, los de mayor error y algunos al azar, con su primer frame, el valor real y el predicho.
El error de un plato es la media, sobre los cinco objetivos, de |error| / media real del objetivo en test (como el PMAE).
Necesita los frames en `data/` (en local o en el pod).

In [ ]:
def gallery(run, count=4, seed=0):
    try:
        from nutrition5k import load_dataset

        dataset = load_dataset(Path("data/nutrition5k"), full=SPLIT == "full", frame_mode=MODE)
    except Exception as exc:  # no local data, e.g. on Colab reading only Drive
        print(f"the gallery needs the frames in data/: {exc}")
        return
    records = {record.dish_id: record for record in dataset.test if record.frame_paths}
    predicted = pd.read_csv(Path("outputs/mobilenet") / run / "predictions.csv").set_index("dish_id")
    predicted = predicted.loc[predicted.index.intersection(list(records))]
    truth = pd.DataFrame({dish: records[dish].targets for dish in predicted.index}, index=TARGETS).T
    error = ((predicted[list(TARGETS)] - truth).abs() / truth.mean()).mean(axis=1).sort_values()
    rest = error.iloc[count:-count]
    groups = {
        "lowest error": error.index[:count],
        "highest error": error.index[-count:][::-1],
        "random": rest.sample(min(count, len(rest)), random_state=seed).index,
    }
    fig, axes = plt.subplots(len(groups), count, figsize=(3.6 * count, 3.4 * len(groups)), squeeze=False)
    for row, (title, dishes) in zip(axes, groups.items()):
        for ax, dish in zip(row, dishes):
            ax.imshow(plt.imread(records[dish].frame_paths[0]))
            lines = [
                f"{target.removeprefix('total_')}: {truth.loc[dish, target]:.0f} → {predicted.loc[dish, target]:.0f}"
                for target in TARGETS
            ]
            ax.set_title(f"{title}\n{dish}", loc="left", color=INK, fontsize=9)
            ax.set_xlabel("\n".join(lines) + f"\nerror {error[dish]:.0%}", color=MUTED, fontsize=8, loc="left")
            ax.set_xticks([]), ax.set_yticks([])
        for ax in row[len(dishes):]:
            ax.axis("off")
    fig.suptitle(f"{run}: real → predicted (g, kcal)", x=0.01, ha="left", color=INK)
    plt.tight_layout()
    plt.show()


gallery(RUN)  # RUN se define arriba (el mejor por validación); se puede escribir otro nombre